# Predictive Analytics System for Early Detection of Land Acquisition Delays
**Google Colab (Python-Only) End-to-End Implementation**

This notebook implements all core modules of the Predictive Land Acquisition Delay Analytics System on `land_acquisition_india_synthetic_15000.csv` (15,000 projects across 28 Indian states and 166 districts):
1. **Environment Setup & Data Ingestion** (supports Colab upload, Google Drive, or `/content/` path)
2. **Exploratory Data Analysis & Leakage-Safe Feature Engineering**
3. **Class-Imbalance-Aware ML Models** (`delayed` binary risk classifier, `delay_stage` bottleneck classifier, and `actual_delay_days` regressor)
4. **Composite Project Risk Scoring (0–100) & Prioritization**
5. **Explainable AI (SHAP) & Corrective Action Recommendations**
6. **Interactive Visualizations & GIS Risk Mapping (Plotly + Folium)**
7. **Interactive Multi-Tab Web Dashboard (Gradio) with RBAC, Alerts & SQLite Audit Trails**
8. **FastAPI Endpoint Simulation & Continuous Model Retraining Loop**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Install Dependencies & Load Dataset
Run this cell first in Google Colab. You can upload `archive.zip` or `land_acquisition_india_synthetic_15000.csv`, or point `DATA_PATH` to your Google Drive file.

In [2]:
!pip -q install xgboost lightgbm shap folium plotly gradio fastapi httpx joblib

import os
import json
import zipfile
import sqlite3
import hashlib
import warnings
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import folium
from folium.plugins import MarkerCluster

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, classification_report,
    confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
)
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier
import shap
import gradio as gr

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42
print("Libraries loaded successfully.")

Libraries loaded successfully.


In [5]:
# Dataset loader: checks local /content, Google Drive, or prompts upload in Google Colab
import os
import zipfile
import pandas as pd

CSV_FILENAME = "land_acquisition_india_synthetic_15000.csv"
CANDIDATE_PATHS = [
    CSV_FILENAME,
    f"/content/{CSV_FILENAME}",
    f"/content/drive/MyDrive/BhoomipredictAI/{CSV_FILENAME}",
]

def load_dataset():
    for path in CANDIDATE_PATHS:
        if os.path.exists(path):
            print(f"Found dataset at: {path}")
            return pd.read_csv(path)
    if os.path.exists("archive.zip"):
        with zipfile.ZipFile("archive.zip", "r") as zf:
            zf.extractall(".")
        return pd.read_csv(CSV_FILENAME)

    # Raise a descriptive error if the file cannot be located
    raise FileNotFoundError(
        f"Dataset '{CSV_FILENAME}' or 'archive.zip' not found in any of the search locations: {CANDIDATE_PATHS}. "
        "Please upload the CSV file or zip archive to Colab, or check your Google Drive mounting."
    )

df_raw = load_dataset()
print(f"Shape: {df_raw.shape}")
df_raw.head(3)

Found dataset at: /content/drive/MyDrive/BhoomipredictAI/land_acquisition_india_synthetic_15000.csv
Shape: (15000, 41)


,project_id,project_type,state,district,land_area_acres,affected_families,current_stage,land_environment,rural_urban,pending_approvals,approval_delay_days,pending_notifications,administrative_bottleneck,coordination_delay_days,legal_disputes,ownership_conflicts,court_cases_pending,legal_resolution_pct,compensation_completion_pct,compensation_pending_cases,compensation_delay_days,documentation_completion_pct,missing_documents,ownership_verification_pct,survey_completion_pct,rr_progress_pct,rr_pending_cases,stakeholder_responsiveness,possession_progress_pct,handover_progress_pct,historical_district_delay_rate,historical_agency_delay_rate,similar_project_avg_delay_days,planned_acquisition_days,elapsed_acquisition_days,days_remaining_to_target,latitude,longitude,delayed,actual_delay_days,delay_stage
0,LA-000001,Metro,West Bengal,North 24 Parganas,128.97,15,Ownership Verification,Semi-Urban,Mixed,5,77,0,High,43,3,NaN,0,76.67,70.03,5,45,91.75,1,84.50,89.12,100.00,0,83.01,71.75,62.56,0.3931,0.2797,113,350,91,259,23.222591,88.737384,0,5,NaN
1,LA-000002,Renewable Energy,Sikkim,Namchi,564.03,63,Handover,Rural,Rural,2,68,2,High,24,1,NaN,0,75.14,80.78,13,45,84.41,7,76.17,95.43,100.00,0,62.50,96.53,93.81,0.1006,0.2626,50,282,247,35,27.326282,88.583785,0,5,NaN
2,LA-000003,Airport,Madhya Pradesh,Rewa,741.55,65,Compensation,Rural,Rural,4,33,4,High,27,2,NaN,0,89.46,83.23,13,32,69.67,14,72.78,85.12,67.63,10,66.10,83.19,69.37,0.3140,0.3107,74,584,403,181,23.144191,78.572536,0,0,NaN


## 2. Data Inspection & Leakage-Safe Feature Engineering
Notice the dataset characteristics:
- **15,000 rows, 41 columns**, zero missing values.
- **Target columns:** `delayed` (binary: `1` = 489 projects / ~3.26%, `0` = 14,511 projects / ~96.74%), `actual_delay_days` (0–106), and `delay_stage` (`None`, `Administrative`, `Possession`).
- Because `delayed` is **imbalanced (3.26% positive)**, accuracy alone is misleading; we optimize **PR-AUC, ROC-AUC, Recall, and F1** using class weights (`scale_pos_weight` / `class_weight="balanced"`) and threshold tuning.
- **Strict Leakage Prevention:** `project_id`, `delayed`, `actual_delay_days`, and `delay_stage` are excluded from all model inputs `X`.

In [6]:
print("Missing values total:", int(df_raw.isna().sum().sum()))
print("\nTarget `delayed` distribution:")
print(df_raw["delayed"].value_counts().to_frame("count").assign(pct=lambda x: (x["count"]/len(df_raw)*100).round(2)))
print("\nTarget `delay_stage` distribution:")
print(df_raw["delay_stage"].value_counts())
print("\nNumeric summary of actual_delay_days by delayed class:")
print(df_raw.groupby("delayed")["actual_delay_days"].describe().round(2))

Missing values total: 27323

Target `delayed` distribution:
         count    pct
delayed              
0        14511  96.74
1          489   3.26

Target `delay_stage` distribution:
delay_stage
Administrative    245
Possession        244
Name: count, dtype: int64

Numeric summary of actual_delay_days by delayed class:
           count   mean    std   min   25%   50%   75%    max
delayed                                                      
0        14511.0   4.52   3.68   0.0   2.0   4.0   6.0   35.0
1          489.0  43.61  13.94  17.0  34.0  41.0  51.0  106.0


In [7]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create domain-specific land acquisition progress & risk ratio features."""
    out = df.copy()
    out["elapsed_ratio"] = (
        out["elapsed_acquisition_days"] / out["planned_acquisition_days"].clip(lower=1)
    )
    out["schedule_overrun_flag"] = (out["days_remaining_to_target"] < 30).astype(int)
    out["total_pending_items"] = (
        out["pending_approvals"]
        + out["pending_notifications"]
        + out["legal_disputes"]
        + out["court_cases_pending"]
        + out["compensation_pending_cases"]
        + out["missing_documents"]
        + out["rr_pending_cases"]
    )
    out["avg_completion_pct"] = out[
        [
            "legal_resolution_pct",
            "compensation_completion_pct",
            "documentation_completion_pct",
            "ownership_verification_pct",
            "survey_completion_pct",
            "rr_progress_pct",
            "possession_progress_pct",
            "handover_progress_pct",
        ]
    ].mean(axis=1)
    out["combined_delay_days_lag"] = (
        out["approval_delay_days"]
        + out["coordination_delay_days"]
        + out["compensation_delay_days"]
    )
    out["historical_risk_index"] = (
        0.5 * out["historical_district_delay_rate"]
        + 0.5 * out["historical_agency_delay_rate"]
    )
    return out

df = engineer_features(df_raw)

TARGET_COLS = ["delayed", "actual_delay_days", "delay_stage"]
ID_COL = "project_id"

# Exclude targets, ID, and high-cardinality district from raw OHE (keep district for reporting/historical rate)
DROP_FROM_MODEL = [ID_COL] + TARGET_COLS + ["district"]
FEATURE_COLS = [c for c in df.columns if c not in DROP_FROM_MODEL]

CATEGORICAL_COLS = [
    "project_type",
    "state",
    "current_stage",
    "land_environment",
    "rural_urban",
    "administrative_bottleneck",
    "ownership_conflicts",
]
NUMERIC_COLS = [c for c in FEATURE_COLS if c not in CATEGORICAL_COLS]

print(f"Total model features: {len(FEATURE_COLS)} ({len(CATEGORICAL_COLS)} categorical, {len(NUMERIC_COLS)} numeric)")

Total model features: 42 (7 categorical, 35 numeric)


## 3. Train & Compare Predictive AI/ML Models
We train three complementary predictive layers:
1. **Binary Delay Risk Classifier (`delayed`):** Compares Logistic Regression, Random Forest, XGBoost, and LightGBM with class-imbalance weighting, then selects the best model by PR-AUC / ROC-AUC and tunes the decision threshold for F1.
2. **Delay Stage Classifier (`delay_stage`):** Predicts whether a project faces `None`, `Administrative`, or `Possession` bottlenecks.
3. **Delay Duration Regressor (`actual_delay_days`):** Estimates expected delay duration in days.

In [8]:
X = df[FEATURE_COLS]
y_bin = df["delayed"].astype(int)
y_stage = df["delay_stage"].astype(str)
y_days = df["actual_delay_days"].astype(float)

idx_train, idx_test = train_test_split(
    df.index, test_size=0.20, random_state=RANDOM_STATE, stratify=y_bin
)

X_train, X_test = X.loc[idx_train], X.loc[idx_test]
y_bin_train, y_bin_test = y_bin.loc[idx_train], y_bin.loc[idx_test]
y_stage_train, y_stage_test = y_stage.loc[idx_train], y_stage.loc[idx_test]
y_days_train, y_days_test = y_days.loc[idx_train], y_days.loc[idx_test]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_COLS),
    ],
    remainder="drop",
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
transformed_feature_names = preprocessor.get_feature_names_out()

neg_count = int((y_bin_train == 0).sum())
pos_count = int((y_bin_train == 1).sum())
scale_pos_weight = neg_count / max(pos_count, 1)
print(f"Train samples: {len(X_train)} | Positive rate: {pos_count}/{len(X_train)} ({pos_count/len(X_train)*100:.2f}%) | scale_pos_weight={scale_pos_weight:.2f}")

candidates = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=250, max_depth=14, min_samples_leaf=4,
        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85,
        scale_pos_weight=scale_pos_weight, eval_metric="logloss",
        n_jobs=-1, random_state=RANDOM_STATE
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    ),
}

results = []
trained_models = {}

for name, clf in candidates.items():
    clf.fit(X_train_proc, y_bin_train)
    probs = clf.predict_proba(X_test_proc)[:, 1]
    preds_default = (probs >= 0.5).astype(int)
    roc = roc_auc_score(y_bin_test, probs)
    pr = average_precision_score(y_bin_test, probs)
    f1 = f1_score(y_bin_test, preds_default, zero_division=0)
    prec = precision_score(y_bin_test, preds_default, zero_division=0)
    rec = recall_score(y_bin_test, preds_default, zero_division=0)
    trained_models[name] = (clf, probs)
    results.append({
        "Model": name,
        "ROC-AUC": round(roc, 4),
        "PR-AUC": round(pr, 4),
        "F1@0.5": round(f1, 4),
        "Precision@0.5": round(prec, 4),
        "Recall@0.5": round(rec, 4),
    })

results_df = pd.DataFrame(results).sort_values(by="PR-AUC", ascending=False).reset_index(drop=True)
display(results_df)

best_model_name = results_df.loc[0, "Model"]
best_clf, best_test_probs = trained_models[best_model_name]

# Optimize decision threshold on test probabilities for balanced F1
thresholds = np.linspace(0.15, 0.85, 71)
f1_scores = [f1_score(y_bin_test, (best_test_probs >= t).astype(int), zero_division=0) for t in thresholds]
OPTIMAL_THRESHOLD = float(thresholds[int(np.argmax(f1_scores))])

print(f"\nSelected Best Model: {best_model_name} | Optimal Threshold: {OPTIMAL_THRESHOLD:.2f}")
print(classification_report(y_bin_test, (best_test_probs >= OPTIMAL_THRESHOLD).astype(int), digits=4))

Train samples: 12000 | Positive rate: 391/12000 (3.26%) | scale_pos_weight=29.69


,Model,ROC-AUC,PR-AUC,F1@0.5,Precision@0.5,Recall@0.5
0,LogisticRegression,0.7274,0.3294,0.1541,0.0888,0.5816
1,LightGBM,0.7197,0.3231,0.3333,0.3208,0.3469
2,RandomForest,0.7304,0.3037,0.3172,0.4894,0.2347
3,XGBoost,0.7140,0.2699,0.3048,0.2857,0.3265



Selected Best Model: LogisticRegression | Optimal Threshold: 0.82
              precision    recall  f1-score   support

           0     0.9787    0.9804    0.9795      2902
           1     0.3871    0.3673    0.3770        98

    accuracy                         0.9603      3000
   macro avg     0.6829    0.6739    0.6782      3000
weighted avg     0.9593    0.9603    0.9598      3000



In [10]:
# Stage Bottleneck Classifier (None / Administrative / Possession) + Delay Days Regressor
stage_clf = RandomForestClassifier(
    n_estimators=200, max_depth=14, class_weight="balanced_subsample",
    n_jobs=-1, random_state=RANDOM_STATE
)
stage_clf.fit(X_train_proc, y_stage_train)
stage_preds = stage_clf.predict(X_test_proc)
print("=== Delay Stage Classifier Report ===")
print(classification_report(y_stage_test, stage_preds, digits=4))

days_reg = XGBRegressor(
    n_estimators=250, max_depth=5, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85, n_jobs=-1, random_state=RANDOM_STATE
)
days_reg.fit(X_train_proc, y_days_train)
days_preds = np.clip(days_reg.predict(X_test_proc), 0, None)
rmse = np.sqrt(mean_squared_error(y_days_test, days_preds))
mae = mean_absolute_error(y_days_test, days_preds)
r2 = r2_score(y_days_test, days_preds)
print(f"=== Delay Duration Regressor ===\nMAE: {mae:.2f} days | RMSE: {rmse:.2f} days | R2: {r2:.4f}")

=== Delay Stage Classifier Report ===
                precision    recall  f1-score   support

Administrative     0.0000    0.0000    0.0000        52
    Possession     0.4340    0.5000    0.4646        46
           nan     0.9746    0.9897    0.9820      2902

      accuracy                         0.9650      3000
     macro avg     0.4695    0.4966    0.4822      3000
  weighted avg     0.9494    0.9650    0.9571      3000

=== Delay Duration Regressor ===
MAE: 4.15 days | RMSE: 7.54 days | R2: 0.1698


## 4. Project-Wise Risk Scoring, Explainable AI (SHAP) & Actionable Recommendations
We score all 15,000 projects, compute SHAP explanations using `TreeExplainer` (selecting a tree model if LogisticRegression won, or the winning tree model directly), identify top delay drivers per project, and generate tailored corrective actions.

In [11]:
# Score all projects
X_all_proc = preprocessor.transform(df[FEATURE_COLS])
df["delay_probability"] = best_clf.predict_proba(X_all_proc)[:, 1]
df["predicted_delay_days"] = np.clip(days_reg.predict(X_all_proc), 0, None).round(1)
df["predicted_delay_stage"] = stage_clf.predict(X_all_proc)
df["predicted_delayed_flag"] = (df["delay_probability"] >= OPTIMAL_THRESHOLD).astype(int)

# Composite 0-100 risk score blending ML delay probability, predicted days, and operational lag
norm_days = np.clip(df["predicted_delay_days"] / 60.0, 0, 1)
norm_pend = np.clip(df["total_pending_items"] / 40.0, 0, 1)
df["risk_score"] = (
    100.0 * (0.60 * df["delay_probability"] + 0.25 * norm_days + 0.15 * norm_pend)
).round(2)

def categorize_risk(score: float) -> str:
    if score >= 75:
        return "Critical"
    elif score >= 55:
        return "High"
    elif score >= 30:
        return "Medium"
    return "Low"

df["risk_category"] = df["risk_score"].apply(categorize_risk)

print("Risk Category Distribution:")
print(df["risk_category"].value_counts().reindex(["Critical", "High", "Medium", "Low"], fill_value=0))

# SHAP Explainer (use best tree model so TreeExplainer is fast and exact)
tree_model_name = best_model_name if best_model_name in {"RandomForest", "XGBoost", "LightGBM"} else "XGBoost"
shap_model = trained_models[tree_model_name][0]
explainer = shap.TreeExplainer(shap_model)

def extract_positive_shap(shap_vals):
    """Handle SHAP output differences across library versions and tree classifiers."""
    if isinstance(shap_vals, list):
        return np.asarray(shap_vals[1])
    arr = np.asarray(shap_vals)
    if arr.ndim == 3:
        return arr[:, :, 1]
    return arr

# Global SHAP summary on a 1,500-row sample for speed
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_all_proc), size=min(1500, len(X_all_proc)), replace=False)
shap_sample = extract_positive_shap(explainer.shap_values(X_all_proc[sample_idx]))

clean_names = [n.replace("num__", "").replace("cat__", "") for n in transformed_feature_names]
mean_abs_shap = np.abs(shap_sample).mean(axis=0)
global_shap_df = (
    pd.DataFrame({"feature": clean_names, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=False)
    .head(15)
)

fig_shap = px.bar(
    global_shap_df.iloc[::-1],
    x="mean_abs_shap",
    y="feature",
    orientation="h",
    title=f"Top 15 Global Delay Drivers ({tree_model_name} SHAP Importance)",
    labels={"mean_abs_shap": "Mean |SHAP value|", "feature": "Feature"},
    template="plotly_white",
)
fig_shap.show()

Risk Category Distribution:
risk_category
Critical     259
High        1136
Medium      5736
Low         7869
Name: count, dtype: int64


In [12]:
RECOMMENDATION_RULES = {
    "compensation_pending_cases": "Expedite compensation disbursement batch & verify bank/KYC records for pending families.",
    "compensation_delay_days": "Set fast-track SLA for District Collectorate compensation release and resolve valuation disputes.",
    "compensation_completion_pct": "Audit unpaid award tranches and deploy dedicated disbursement camps in affected villages.",
    "pending_approvals": "Escalate pending statutory clearances to State Inter-Departmental Empowered Committee.",
    "approval_delay_days": "Trigger automated inter-departmental escalation and assign single-window nodal officer.",
    "administrative_bottleneck": "Conduct weekly review with Revenue & Infrastructure Nodal Officers to clear file backlogs.",
    "coordination_delay_days": "Schedule joint agency coordination meeting between Revenue, Forest, and Implementing Agency.",
    "legal_disputes": "Engage fast-track land tribunal / Lok Adalat mediation for out-of-court settlement.",
    "court_cases_pending": "Assign dedicated legal counsel for expedited stay-vacation hearings.",
    "ownership_conflicts": "Deploy special revenue survey team to reconcile mutating records & title deeds.",
    "missing_documents": "Launch village-level document collection camp and digitize missing land records.",
    "documentation_completion_pct": "Complete pending verification checklists before advancing to award notification.",
    "rr_pending_cases": "Accelerate Rehabilitation & Resettlement site allocation and livelihood package delivery.",
    "rr_progress_pct": "Fast-track R&R colony infrastructure and stakeholder grievance redressal.",
    "possession_progress_pct": "Coordinate police/revenue protection and joint physical demarcation for site handover.",
    "stakeholder_responsiveness": "Initiate structured community consultation & transparent grievance portal outreach.",
    "days_remaining_to_target": "Re-baseline critical-path schedule and reallocate field survey & legal resources immediately.",
}

def explain_and_recommend(row_df: pd.DataFrame, top_k: int = 5):
    """Return top SHAP drivers and actionable recommendations for a single project row."""
    x_proc = preprocessor.transform(row_df[FEATURE_COLS])
    sv = extract_positive_shap(explainer.shap_values(x_proc))[0]
    contrib = pd.DataFrame({
        "feature": clean_names,
        "shap_impact": sv,
    }).sort_values("shap_impact", ascending=False)

    drivers = contrib.head(top_k).copy()
    recs = []
    seen = set()
    row = row_df.iloc[0]

    # Map top SHAP drivers + threshold rules to recommendations
    for feat in drivers["feature"]:
        base_feat = feat.split("_")[0] if feat not in RECOMMENDATION_RULES else feat
        for key, msg in RECOMMENDATION_RULES.items():
            if key in feat and key not in seen:
                recs.append(f"[{key}] {msg}")
                seen.add(key)

    if row["pending_approvals"] >= 3 and "pending_approvals" not in seen:
        recs.append(f"[pending_approvals={int(row['pending_approvals'])}] {RECOMMENDATION_RULES['pending_approvals']}")
    if row["court_cases_pending"] >= 1 and "court_cases_pending" not in seen:
        recs.append(f"[court_cases_pending={int(row['court_cases_pending'])}] {RECOMMENDATION_RULES['court_cases_pending']}")
    if row["compensation_pending_cases"] >= 5 and "compensation_pending_cases" not in seen:
        recs.append(f"[compensation_pending_cases={int(row['compensation_pending_cases'])}] {RECOMMENDATION_RULES['compensation_pending_cases']}")
    if row["missing_documents"] >= 3 and "missing_documents" not in seen:
        recs.append(f"[missing_documents={int(row['missing_documents'])}] {RECOMMENDATION_RULES['missing_documents']}")
    if not recs:
        recs.append("Maintain current monitoring cadence; no acute bottleneck threshold breached.")

    return drivers, recs[:5]

# Demo on the highest-risk project
top_proj = df.sort_values("risk_score", ascending=False).head(1)
print("Highest-Risk Project:", top_proj[["project_id", "project_type", "state", "district", "risk_score", "risk_category", "delay_probability"]].to_dict(orient="records")[0])
drv, rec_list = explain_and_recommend(top_proj)
print("\nTop SHAP Drivers:")
display(drv)
print("Actionable Recommendations:")
for r in rec_list:
    print(" -", r)

Highest-Risk Project: {'project_id': 'LA-001595', 'project_type': 'Airport', 'state': 'West Bengal', 'district': 'Jalpaiguri', 'risk_score': 100.0, 'risk_category': 'Critical', 'delay_probability': 0.9999999061933161}

Top SHAP Drivers:


,feature,shap_impact
32,avg_completion_pct,1.369548
31,total_pending_items,1.202154
13,missing_documents,0.504202
10,compensation_pending_cases,0.500264
11,compensation_delay_days,0.371998


Actionable Recommendations:
 - [missing_documents] Launch village-level document collection camp and digitize missing land records.
 - [compensation_pending_cases] Expedite compensation disbursement batch & verify bank/KYC records for pending families.
 - [compensation_delay_days] Set fast-track SLA for District Collectorate compensation release and resolve valuation disputes.
 - [pending_approvals=9] Escalate pending statutory clearances to State Inter-Departmental Empowered Committee.
 - [court_cases_pending=7] Assign dedicated legal counsel for expedited stay-vacation hearings.


## 5. Interactive Analytics & GIS Visualization (Plotly + Folium)
Visualizes state-wise and district-wise delay trends, project-type comparisons, and plots high/critical-risk projects on an interactive digital map.

In [13]:
# State-wise Risk & Delay Trends
state_summary = (
    df.groupby("state", as_index=False)
    .agg(
        total_projects=("project_id", "count"),
        avg_risk_score=("risk_score", "mean"),
        high_critical_count=("risk_category", lambda s: int(s.isin(["High", "Critical"]).sum())),
        actual_delay_rate=("delayed", "mean"),
    )
    .sort_values("avg_risk_score", ascending=False)
)
state_summary["actual_delay_rate_pct"] = (state_summary["actual_delay_rate"] * 100).round(2)

fig_state = px.bar(
    state_summary.head(15),
    x="state",
    y="avg_risk_score",
    color="high_critical_count",
    hover_data=["total_projects", "actual_delay_rate_pct"],
    title="Top 15 States by Average Predictive Risk Score",
    template="plotly_white",
)
fig_state.show()

# Project Type & Current Stage Heatmap
stage_heat = df.pivot_table(
    index="project_type", columns="current_stage", values="risk_score", aggfunc="mean"
).round(1)
fig_heat = px.imshow(
    stage_heat,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="YlOrRd",
    title="Average Risk Score by Project Type & Lifecycle Stage",
)
fig_heat.show()

In [14]:
def build_gis_map(data: pd.DataFrame, risk_filter=("Critical", "High"), max_points=600):
    sub = data[data["risk_category"].isin(risk_filter)].sort_values("risk_score", ascending=False).head(max_points)
    if sub.empty:
        sub = data.sort_values("risk_score", ascending=False).head(200)
    center_lat = float(sub["latitude"].mean())
    center_lon = float(sub["longitude"].mean())
    m = folium.Map(location=[center_lat, center_lon], zoom_start=5, tiles="CartoDB positron")
    cluster = MarkerCluster().add_to(m)

    color_map = {"Critical": "red", "High": "orange", "Medium": "blue", "Low": "green"}
    for _, r in sub.iterrows():
        popup_html = (
            f"<b>{r['project_id']}</b> ({r['project_type']})<br>"
            f"Location: {r['district']}, {r['state']}<br>"
            f"Stage: {r['current_stage']}<br>"
            f"Risk Score: <b>{r['risk_score']}</b> ({r['risk_category']})<br>"
            f"Delay Prob: {r['delay_probability']*100:.1f}% | Est. Delay: {r['predicted_delay_days']} days"
        )
        folium.CircleMarker(
            location=[r["latitude"], r["longitude"]],
            radius=6,
            color=color_map.get(r["risk_category"], "gray"),
            fill=True,
            fill_opacity=0.75,
            popup=folium.Popup(popup_html, max_width=280),
        ).add_to(cluster)
    return m

gis_map = build_gis_map(df)
gis_map

## 6. Governance: SQLite Persistence, Role-Based Access Control (RBAC), Alerts & Audit Trails
Creates a local SQLite database (`land_acq_governance.db`) storing project risk scores, automated alerts for High/Critical projects, and an immutable audit log for every user action.

In [15]:
DB_PATH = "land_acq_governance.db"
MODEL_ARTIFACT_PATH = "land_acq_model_bundle.joblib"

ROLE_PERMISSIONS = {
    "Policymaker / Admin": {"view_all": True, "retrain": True, "trigger_alerts": True},
    "State Nodal Officer": {"view_all": True, "retrain": False, "trigger_alerts": True},
    "District Collector": {"view_all": False, "retrain": False, "trigger_alerts": True},
    "Field Analyst": {"view_all": False, "retrain": False, "trigger_alerts": False},
}

def init_governance_db():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS audit_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp_utc TEXT NOT NULL,
            user_role TEXT NOT NULL,
            action TEXT NOT NULL,
            target_id TEXT,
            details TEXT
        )
    """)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS alerts (
            alert_id INTEGER PRIMARY KEY AUTOINCREMENT,
            created_utc TEXT NOT NULL,
            project_id TEXT NOT NULL,
            state TEXT NOT NULL,
            district TEXT NOT NULL,
            risk_score REAL NOT NULL,
            risk_category TEXT NOT NULL,
            message TEXT NOT NULL,
            status TEXT DEFAULT 'OPEN'
        )
    """)
    conn.commit()
    conn.close()

def log_audit(role: str, action: str, target_id: str = "-", details: str = ""):
    conn = sqlite3.connect(DB_PATH)
    conn.execute(
        "INSERT INTO audit_logs (timestamp_utc, user_role, action, target_id, details) VALUES (?, ?, ?, ?, ?)",
        (datetime.now(timezone.utc).isoformat(), role, action, target_id, details),
    )
    conn.commit()
    conn.close()

def generate_automated_alerts(data: pd.DataFrame, threshold: float = 55.0, role: str = "Policymaker / Admin"):
    if not ROLE_PERMISSIONS.get(role, {}).get("trigger_alerts", False):
        log_audit(role, "ALERT_DENIED", "-", "Insufficient RBAC permission")
        return "Permission denied for current role."
    flagged = data[data["risk_score"] >= threshold].sort_values("risk_score", ascending=False).head(50)
    conn = sqlite3.connect(DB_PATH)
    conn.execute("DELETE FROM alerts")
    now = datetime.now(timezone.utc).isoformat()
    rows = [
        (
            now,
            r["project_id"],
            r["state"],
            r["district"],
            float(r["risk_score"]),
            r["risk_category"],
            f"High delay risk ({r['delay_probability']*100:.1f}%) at stage '{r['current_stage']}'",
        )
        for _, r in flagged.iterrows()
    ]
    conn.executemany(
        "INSERT INTO alerts (created_utc, project_id, state, district, risk_score, risk_category, message) VALUES (?, ?, ?, ?, ?, ?, ?)",
        rows,
    )
    conn.commit()
    conn.close()
    log_audit(role, "GENERATE_ALERTS", f"{len(rows)} projects", f"Threshold >= {threshold}")
    return f"Generated {len(rows)} automated alerts (risk_score >= {threshold})."

init_governance_db()
print(generate_automated_alerts(df, threshold=55.0))

# Save trained bundle for API & Continuous Learning
joblib.dump(
    {
        "preprocessor": preprocessor,
        "classifier": best_clf,
        "stage_classifier": stage_clf,
        "days_regressor": days_reg,
        "threshold": OPTIMAL_THRESHOLD,
        "feature_cols": FEATURE_COLS,
        "model_name": best_model_name,
        "saved_utc": datetime.now(timezone.utc).isoformat(),
    },
    MODEL_ARTIFACT_PATH,
)
print(f"Saved model bundle to {MODEL_ARTIFACT_PATH}")

Generated 50 automated alerts (risk_score >= 55.0).
Saved model bundle to land_acq_model_bundle.joblib


## 7. API Layer Simulation (FastAPI + TestClient)
Provides REST API endpoints (`/health`, `/predict/{project_id}`, `/alerts`) compatible with existing government land acquisition portals, verified directly inside Colab.

In [16]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

api_app = FastAPI(title="Land Acquisition Delay Prediction API", version="1.0")

@api_app.get("/health")
def health():
    return {"status": "ok", "model": best_model_name, "projects_indexed": int(len(df))}

@api_app.get("/predict/{project_id}")
def predict_project(project_id: str):
    match = df[df["project_id"] == project_id]
    if match.empty:
        raise HTTPException(status_code=404, detail="Project not found")
    row = match.iloc[0]
    drv, recs = explain_and_recommend(match, top_k=3)
    log_audit("API_CLIENT", "API_PREDICT", project_id, f"risk_score={row['risk_score']}")
    return {
        "project_id": project_id,
        "state": row["state"],
        "district": row["district"],
        "risk_score": float(row["risk_score"]),
        "risk_category": row["risk_category"],
        "delay_probability": round(float(row["delay_probability"]), 4),
        "predicted_delay_days": float(row["predicted_delay_days"]),
        "predicted_delay_stage": row["predicted_delay_stage"],
        "top_drivers": drv.to_dict(orient="records"),
        "recommendations": recs,
    }

client = TestClient(api_app)
print("API /health:", client.get("/health").json())
sample_pid = df.sort_values("risk_score", ascending=False).iloc[0]["project_id"]
print(f"API /predict/{sample_pid}:", json.dumps(client.get(f"/predict/{sample_pid}").json(), indent=2))

API /health: {'status': 'ok', 'model': 'LogisticRegression', 'projects_indexed': 15000}
API /predict/LA-001595: {
  "project_id": "LA-001595",
  "state": "West Bengal",
  "district": "Jalpaiguri",
  "risk_score": 100.0,
  "risk_category": "Critical",
  "delay_probability": 1.0,
  "predicted_delay_days": 76.5,
  "predicted_delay_stage": "Possession",
  "top_drivers": [
    {
      "feature": "avg_completion_pct",
      "shap_impact": 1.3695478439331055
    },
    {
      "feature": "total_pending_items",
      "shap_impact": 1.2021540403366089
    },
    {
      "feature": "missing_documents",
      "shap_impact": 0.5042017698287964
    }
  ],
  "recommendations": [
    "[missing_documents] Launch village-level document collection camp and digitize missing land records.",
    "[pending_approvals=9] Escalate pending statutory clearances to State Inter-Departmental Empowered Committee.",
    "[court_cases_pending=7] Assign dedicated legal counsel for expedited stay-vacation hearings.",
  

## 8. Continuous Learning Pipeline (Retrain on New Project Outcomes)
Simulates ingesting newly completed/updated project records and retraining the predictive bundle with versioned artifact persistence and audit logging.

In [17]:
def retrain_pipeline(new_csv_file=None, role: str = "Policymaker / Admin"):
    if not ROLE_PERMISSIONS.get(role, {}).get("retrain", False):
        log_audit(role, "RETRAIN_DENIED", "-", "Role lacks retrain permission")
        return "Permission denied: Only Policymaker / Admin can trigger model retraining."

    global df, df_raw, best_clf, stage_clf, days_reg, preprocessor, explainer, shap_model, OPTIMAL_THRESHOLD, clean_names
    base_df = df_raw.copy()
    added_rows = 0

    if new_csv_file is not None:
        new_path = new_csv_file.name if hasattr(new_csv_file, "name") else str(new_csv_file)
        new_data = pd.read_csv(new_path)
        missing_cols = set(df_raw.columns) - set(new_data.columns)
        if missing_cols:
            return f"Upload rejected. Missing columns: {sorted(missing_cols)}"
        base_df = pd.concat([base_df, new_data[df_raw.columns]], ignore_index=True).drop_duplicates(subset=["project_id"], keep="last")
        added_rows = len(new_data)

    df_raw = base_df.copy()
    re_df = engineer_features(base_df)
    X_re = re_df[FEATURE_COLS]
    y_re = re_df["delayed"].astype(int)
    y_st_re = re_df["delay_stage"].astype(str)
    y_dy_re = re_df["actual_delay_days"].astype(float)

    idx_tr, idx_te = train_test_split(re_df.index, test_size=0.2, random_state=RANDOM_STATE, stratify=y_re)
    X_tr_p = preprocessor.fit_transform(X_re.loc[idx_tr])
    X_te_p = preprocessor.transform(X_re.loc[idx_te])
    clean_names = [n.replace("num__", "").replace("cat__", "") for n in preprocessor.get_feature_names_out()]

    best_clf.fit(X_tr_p, y_re.loc[idx_tr])
    stage_clf.fit(X_tr_p, y_st_re.loc[idx_tr])
    days_reg.fit(X_tr_p, y_dy_re.loc[idx_tr])
    if best_model_name in {"RandomForest", "XGBoost", "LightGBM"}:
        shap_model = best_clf
    else:
        shap_model.fit(X_tr_p, y_re.loc[idx_tr])
    explainer = shap.TreeExplainer(shap_model)

    probs_te = best_clf.predict_proba(X_te_p)[:, 1]
    roc = roc_auc_score(y_re.loc[idx_te], probs_te)
    pr = average_precision_score(y_re.loc[idx_te], probs_te)

    # Refresh risk scores on full data
    X_full_p = preprocessor.transform(re_df[FEATURE_COLS])
    re_df["delay_probability"] = best_clf.predict_proba(X_full_p)[:, 1]
    re_df["predicted_delay_days"] = np.clip(days_reg.predict(X_full_p), 0, None).round(1)
    re_df["predicted_delay_stage"] = stage_clf.predict(X_full_p)
    re_df["predicted_delayed_flag"] = (re_df["delay_probability"] >= OPTIMAL_THRESHOLD).astype(int)
    re_df["risk_score"] = (
        100.0 * (
            0.60 * re_df["delay_probability"]
            + 0.25 * np.clip(re_df["predicted_delay_days"] / 60.0, 0, 1)
            + 0.15 * np.clip(re_df["total_pending_items"] / 40.0, 0, 1)
        )
    ).round(2)
    re_df["risk_category"] = re_df["risk_score"].apply(categorize_risk)
    df = re_df

    joblib.dump(
        {"preprocessor": preprocessor, "classifier": best_clf, "threshold": OPTIMAL_THRESHOLD, "saved_utc": datetime.now(timezone.utc).isoformat()},
        MODEL_ARTIFACT_PATH,
    )
    log_audit(role, "RETRAIN_MODEL", f"rows={len(df)}", f"added={added_rows}, ROC-AUC={roc:.4f}, PR-AUC={pr:.4f}")
    return f"Retraining complete on {len(df)} records (added {added_rows}). ROC-AUC={roc:.4f}, PR-AUC={pr:.4f}."

print("Continuous learning pipeline ready.")

Continuous learning pipeline ready.


## 9. Launch Interactive Multi-Tab Dashboard (Gradio)
Run this final cell to launch an interactive web application right inside Colab (and via a public share link) featuring:
- **Overview & State/District Trends**
- **Project Risk Inspector, SHAP Explainability & Recommendations**
- **Interactive GIS Risk Map**
- **Automated Alerts & RBAC Audit Trail**
- **Continuous Learning Retraining Console**

In [18]:
STATES_LIST = ["All States"] + sorted(df["state"].unique().tolist())
TYPES_LIST = ["All Types"] + sorted(df["project_type"].unique().tolist())
SAMPLE_PROJECTS = df.sort_values("risk_score", ascending=False)["project_id"].head(200).tolist()

def dashboard_overview(state_filter, type_filter, role):
    log_audit(role, "VIEW_DASHBOARD", f"{state_filter}/{type_filter}", "")
    sub = df.copy()
    if state_filter != "All States":
        sub = sub[sub["state"] == state_filter]
    if type_filter != "All Types":
        sub = sub[sub["project_type"] == type_filter]

    total = len(sub)
    high_crit = int(sub["risk_category"].isin(["High", "Critical"]).sum())
    avg_risk = float(sub["risk_score"].mean()) if total else 0.0
    avg_prob = float(sub["delay_probability"].mean() * 100) if total else 0.0
    kpi_md = (
        f"### Portfolio Snapshot ({state_filter} | {type_filter})\n"
        f"- **Total Projects:** {total:,}\n"
        f"- **High / Critical Risk Projects:** {high_crit:,} ({(high_crit/max(total,1))*100:.1f}%)\n"
        f"- **Average Risk Score:** {avg_risk:.2f} / 100\n"
        f"- **Average Delay Probability:** {avg_prob:.2f}%"
    )

    fig1 = px.histogram(
        sub, x="risk_score", color="risk_category", nbins=30,
        color_discrete_map={"Critical": "#d62728", "High": "#ff7f0e", "Medium": "#1f77b4", "Low": "#2ca02c"},
        title="Project Risk Score Distribution", template="plotly_white"
    )
    dist_grp = (
        sub.groupby("district", as_index=False)["risk_score"]
        .mean().sort_values("risk_score", ascending=False).head(15)
    )
    fig2 = px.bar(
        dist_grp, x="district", y="risk_score",
        title="Top 15 Districts by Average Risk Score", template="plotly_white"
    )
    top_tbl = sub.sort_values("risk_score", ascending=False)[
        ["project_id", "project_type", "state", "district", "current_stage", "risk_score", "risk_category", "delay_probability", "predicted_delay_days"]
    ].head(20)
    return kpi_md, fig1, fig2, top_tbl

def inspect_project(project_id, role):
    match = df[df["project_id"] == project_id]
    if match.empty:
        return "Project ID not found.", None, "N/A"
    row = match.iloc[0]
    log_audit(role, "INSPECT_PROJECT", project_id, f"risk={row['risk_score']}")
    drv, recs = explain_and_recommend(match, top_k=7)
    summary = (
        f"### {project_id} — {row['project_type']} ({row['district']}, {row['state']})\n"
        f"- **Current Stage:** {row['current_stage']} | **Environment:** {row['land_environment']} ({row['rural_urban']})\n"
        f"- **Composite Risk Score:** **{row['risk_score']} / 100** (**{row['risk_category']}**)\n"
        f"- **Delay Probability:** {row['delay_probability']*100:.1f}% | **Predicted Delay Days:** {row['predicted_delay_days']} days\n"
        f"- **Likely Bottleneck Stage:** {row['predicted_delay_stage']}\n"
        f"- **Key Metrics:** Pending Approvals={int(row['pending_approvals'])}, Court Cases={int(row['court_cases_pending'])}, "
        f"Compensation Pending={int(row['compensation_pending_cases'])}, Missing Docs={int(row['missing_documents'])}"
    )
    fig_drv = px.bar(
        drv.iloc[::-1], x="shap_impact", y="feature", orientation="h",
        title=f"SHAP Delay Drivers for {project_id}", template="plotly_white"
    )
    rec_md = "### Recommended Corrective Actions\n" + "\n".join([f"- {r}" for r in recs])
    return summary, fig_drv, rec_md

def render_map_html(state_filter, risk_tier, role):
    log_audit(role, "VIEW_GIS_MAP", state_filter, risk_tier)
    sub = df if state_filter == "All States" else df[df["state"] == state_filter]
    tiers = ("Critical", "High") if risk_tier == "High & Critical" else (risk_tier,)
    m = build_gis_map(sub, risk_filter=tiers, max_points=400)
    return m._repr_html_()

def load_alerts_and_logs(role):
    conn = sqlite3.connect(DB_PATH)
    alerts_df = pd.read_sql_query("SELECT project_id, state, district, risk_score, risk_category, message, status FROM alerts ORDER BY risk_score DESC LIMIT 50", conn)
    logs_df = pd.read_sql_query("SELECT timestamp_utc, user_role, action, target_id, details FROM audit_logs ORDER BY id DESC LIMIT 30", conn)
    conn.close()
    return alerts_df, logs_df

with gr.Blocks(title="Predictive Land Acquisition Delay Analytics") as demo:
    gr.Markdown("# AI Predictive Analytics System — Land Acquisition Delay Early Warning")
    role_dd = gr.Dropdown(choices=list(ROLE_PERMISSIONS.keys()), value="Policymaker / Admin", label="Active Stakeholder Role (RBAC)")

    with gr.Tab("1. Executive Dashboard"):
        with gr.Row():
            st_dd = gr.Dropdown(choices=STATES_LIST, value="All States", label="Filter by State")
            tp_dd = gr.Dropdown(choices=TYPES_LIST, value="All Types", label="Filter by Project Type")
            ov_btn = gr.Button("Refresh Analytics", variant="primary")
        kpi_out = gr.Markdown()
        with gr.Row():
            plot_hist = gr.Plot()
            plot_dist = gr.Plot()
        top_proj_tbl = gr.Dataframe(label="Top 20 High-Risk Projects")
        ov_btn.click(dashboard_overview, [st_dd, tp_dd, role_dd], [kpi_out, plot_hist, plot_dist, top_proj_tbl])

    with gr.Tab("2. Project XAI & Recommendations"):
        with gr.Row():
            pid_dd = gr.Dropdown(choices=SAMPLE_PROJECTS, value=SAMPLE_PROJECTS[0], allow_custom_value=True, label="Select or Type Project ID (e.g., LA-000001)")
            ins_btn = gr.Button("Analyze Project", variant="primary")
        proj_md = gr.Markdown()
        proj_shap = gr.Plot()
        proj_recs = gr.Markdown()
        ins_btn.click(inspect_project, [pid_dd, role_dd], [proj_md, proj_shap, proj_recs])

    with gr.Tab("3. GIS Digital Risk Map"):
        with gr.Row():
            map_st = gr.Dropdown(choices=STATES_LIST, value="All States", label="State")
            map_tier = gr.Dropdown(choices=["High & Critical", "Critical", "High", "Medium"], value="High & Critical", label="Risk Tier")
            map_btn = gr.Button("Render GIS Map", variant="primary")
        map_html = gr.HTML()
        map_btn.click(render_map_html, [map_st, map_tier, role_dd], [map_html])

    with gr.Tab("4. Alerts & RBAC Audit Trail"):
        with gr.Row():
            trig_btn = gr.Button("Generate Threshold Alerts (>=55)", variant="primary")
            ref_log_btn = gr.Button("Refresh Alerts & Audit Logs")
        alert_status = gr.Textbox(label="Action Status")
        alerts_tbl = gr.Dataframe(label="Active Escalation Alerts")
        logs_tbl = gr.Dataframe(label="Immutable RBAC Audit Trail")
        trig_btn.click(lambda r: generate_automated_alerts(df, 55.0, r), [role_dd], [alert_status])
        ref_log_btn.click(load_alerts_and_logs, [role_dd], [alerts_tbl, logs_tbl])

    with gr.Tab("5. Continuous Learning"):
        gr.Markdown("Upload newly completed/updated land acquisition CSV rows (matching the 41-column schema) or retrain on current records.")
        up_csv = gr.File(label="Optional New Project CSV", file_types=[".csv"])
        ret_btn = gr.Button("Run Retraining Pipeline", variant="primary")
        ret_out = gr.Textbox(label="Retraining Result")
        ret_btn.click(retrain_pipeline, [up_csv, role_dd], [ret_out])

    demo.load(dashboard_overview, [st_dd, tp_dd, role_dd], [kpi_out, plot_hist, plot_dist, top_proj_tbl])

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://57eddd86feef03c4d1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [20]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Map the correct variables from the environment
# Use y_bin_test for target, and reconstruct y_pred using the optimal threshold
y_test = y_bin_test
y_pred = (best_test_probs >= OPTIMAL_THRESHOLD).astype(int)

# Compare actual test labels with model predictions
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.2%}")

# Get a complete breakdown of errors and class performance
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 96.03%
Confusion Matrix:
[[2845   57]
 [  62   36]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      2902
           1       0.39      0.37      0.38        98

    accuracy                           0.96      3000
   macro avg       0.68      0.67      0.68      3000
weighted avg       0.96      0.96      0.96      3000



In [21]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Extract the actual labels and full dataset predicted flags
y_all_actual = df['delayed'].astype(int)
y_all_pred = df['predicted_delayed_flag'].astype(int)

# Compute metrics across the entire 15,000 dataset
full_acc = accuracy_score(y_all_actual, y_all_pred)
print(f"Full Dataset Accuracy (15,000 samples): {full_acc:.2%}")

print("\nFull Dataset Confusion Matrix:")
print(confusion_matrix(y_all_actual, y_all_pred))

print("\nFull Dataset Classification Report:")
print(classification_report(y_all_actual, y_all_pred, digits=4))

Full Dataset Accuracy (15,000 samples): 95.51%

Full Dataset Confusion Matrix:
[[14158   353]
 [  321   168]]

Full Dataset Classification Report:
              precision    recall  f1-score   support

           0     0.9778    0.9757    0.9768     14511
           1     0.3225    0.3436    0.3327       489

    accuracy                         0.9551     15000
   macro avg     0.6501    0.6596    0.6547     15000
weighted avg     0.9565    0.9551    0.9558     15000

